# レッスン 10 - 本番環境でのAIエージェント

このレッスンでは、Microsoft Agent Framework (Python) を使ったAIエージェントの<strong>本番パターン</strong>について学びます。内容は以下の通りです：

- <strong>可観測性</strong> — エージェントのやりとりにタイミングとログを追加する
- <strong>評価</strong> — 評価者エージェントを使用して応答の品質をスコアリングする
- <strong>コスト管理</strong> — トークン最適化とモデル選択のための戦略

シナリオは、旅行計画を支援する<strong>旅行代理店</strong>で、その上にモニタリングと評価がレイヤーとして重ねられています。


## セットアップ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 本番環境における考慮事項

AIエージェントをプロトタイプから本番環境に移行するには、以下の3つの柱に慎重に注意を払う必要があります：

1. <strong>可観測性</strong> — エージェントが何をしているのか、どれくらい時間がかかっているのか、どのツールを呼び出しているのかを把握する必要があります。トレースやログ記録がなければ、本番の問題をデバッグすることはほぼ不可能です。

2. <strong>評価</strong> — 自動化された品質チェックにより、エージェントの応答が時間を経ても正確で完全かつ役立つものであり続けることを保証します。評価エージェントは定義された基準に基づいて応答をスコアリングできます。

3. <strong>コスト管理</strong> — トークン使用量は直接コストに影響します。プロンプトの最適化、モデル選択、キャッシングなどの戦略により、品質を犠牲にせずに費用を抑えることができます。


## オブザーバブルエージェントの構築

旅行ツールを定義し、遅延を監視できるようにタイミングでエージェント呼び出しをラップします。本番環境では、OpenTelemetry や類似のトレーシングバックエンドと統合することになります。


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## 評価パターン

一般的な運用パターンは、二番目のエージェントを<strong>評価者</strong>として使用することです。評価者は、完全性、正確性、役立ち度といった事前定義の基準に対して、一次エージェントの応答を評価します。

これにより以下が可能になります：
- 応答がユーザーに届く前の自動品質ゲート
- プロンプトやモデル変更時の回帰検出
- 時間経過に伴うエージェント性能の継続的監視


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## コスト管理戦略

生産的AIエージェントにおいてコスト管理は非常に重要です。以下が主要な戦略です：

| 戦略 | 説明 |
|---|---|
| <strong>プロンプトの最適化</strong> | システム指示を簡潔に保つ。入力トークンを減らすために冗長なコンテキストを除去する。 |
    "| <strong>モデル選択</strong> | 分類や抽出などの単純なタスクには小型で安価なモデル（例：GPT-5-mini）を使用し、複雑な推論にはより大きなモデルを使う。|\n",
| <strong>キャッシュ</strong> | ツールの結果や頻繁に使うクエリをキャッシュし、冗長なAPI呼び出しを避ける。 |
| <strong>トークン予算</strong> | `max_tokens` を設定して予期せぬ長い応答を防ぐ。 |
| <strong>バッチ処理</strong> | 複数のユーザークエリを可能な限り1つのAPI呼び出しにまとめる。 |

実際には階層化されたアプローチが効果的で、単純なリクエストは高速で安価なモデルにルーティングし、複雑なものだけより高性能なモデルに引き上げます。


## 要約

このレッスンでは、次のことを学びました：

1. タイミングとログ記録を用いてエージェントのやり取りに<strong>可観測性を追加</strong>し、トレースとモニタリングの基盤を整える方法。
2. 完全性、正確性、有用性を評価するエバレータエージェントを使って、エージェントの応答を<strong>自動評価</strong>する方法。
3. プロンプトの最適化、モデル選択、キャッシュ、トークン予算を通じて<strong>コスト管理</strong>を行う方法。

これらの本番用パターンは、AIエージェントを規模に応じて信頼性が高く、測定可能で、コスト効率良くするのに役立ちます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
